In [ ]:
# ============================================================
# EfficientNetV2-M  — pipeline haute performance
# Crop · RandAugment · Mixup · Label Smoothing · Dropout
# Fine-tuning 2 phases · CosineAnnealingWarmRestarts · TTA
# ============================================================
import os, sys, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.transforms import RandAugment
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, ReduceLROnPlateau
from torch.amp import autocast, GradScaler

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 1. Chargement du CSV ─────────────────────────────────────────────────────
df = pd.read_csv("../data/dataset_selection_sans_leger.csv")
df = df.dropna(subset=["path", "label"]).reset_index(drop=True)
df["exists"] = df["path"].apply(os.path.exists)
missing = (~df["exists"]).sum()
if missing:
    print(f"[WARN] {missing} fichiers manquants — retirés.")
    df = df[df["exists"]].reset_index(drop=True)
df = df.drop(columns=["exists"])

print(f"Total images : {len(df)}")
print(df["label"].value_counts())

In [ ]:
# ── 2. Encodage labels + split 80 / 10 / 10 ─────────────────────────────────
le = LabelEncoder()
df["y"] = le.fit_transform(df["label"])
num_classes = len(le.classes_)
print("Classes :", list(le.classes_))

train_df, temp_df = train_test_split(df,      test_size=0.2,  random_state=42, stratify=df["y"])
test_df,  val_df  = train_test_split(temp_df, test_size=0.5,  random_state=42, stratify=temp_df["y"])

for name, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"{name:5s} : {len(d):5d}  |  {dict(d['label'].value_counts())}")

In [ ]:
# ── 3. Crop des bordures noires ───────────────────────────────────────────────
def crop_black_border_pil(img: Image.Image, thr: int = 10, pad: int = 10) -> Image.Image:
    """Retire le fond noir autour du fond d'œil."""
    arr  = np.array(img)
    gray = arr.mean(axis=2)
    mask = gray > thr
    if not mask.any():
        return img
    ys, xs = np.where(mask)
    y0 = max(0, ys.min() - pad)
    y1 = min(arr.shape[0] - 1, ys.max() + pad)
    x0 = max(0, xs.min() - pad)
    x1 = min(arr.shape[1] - 1, xs.max() + pad)
    return img.crop((x0, y0, x1 + 1, y1 + 1))

# Visualisation
sample_paths = df["path"].sample(4, random_state=0).tolist()
fig, axes = plt.subplots(4, 2, figsize=(8, 14))
for row, p in enumerate(sample_paths):
    img = Image.open(p).convert("RGB")
    axes[row, 0].imshow(img);                    axes[row, 0].set_title("Original"); axes[row, 0].axis("off")
    axes[row, 1].imshow(crop_black_border_pil(img)); axes[row, 1].set_title("Cropped");  axes[row, 1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# ── 4. Hyperparamètres ────────────────────────────────────────────────────────
IMG          = 256     # EfficientNetV2-M tourne nativement à 480, 256 est un bon compromis
BATCH        = 16      # Réduit pour EfficientNetV2-M (>50M params)
EPOCHS_P1    = 10      # Phase 1 : backbone gelé
EPOCHS_P2    = 300     # Phase 2 : backbone dégelé
PATIENCE     = 25      # Early stopping patience
LR_P1        = 1e-3   # lr tête seule
LR_P2        = 5e-5   # lr fine-tuning complet
MIXUP_ALPHA  = 0.3    # Mixup (0 = désactivé)
NUM_WORKERS  = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", device)

In [ ]:
# ── 5. Transforms ─────────────────────────────────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    RandAugment(num_ops=2, magnitude=9),   # augmentation automatique forte
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_tfms = transforms.Compose([
    transforms.Resize(IMG + 32),
    transforms.CenterCrop(IMG),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Transforms TTA (5 passes différentes à l'inférence)
tta_tfms_list = [
    eval_tfms,
    transforms.Compose([transforms.Resize(IMG+32), transforms.CenterCrop(IMG),
                        transforms.RandomHorizontalFlip(p=1.0),
                        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize(IMG+32), transforms.CenterCrop(IMG),
                        transforms.RandomVerticalFlip(p=1.0),
                        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize(IMG+32), transforms.CenterCrop(IMG),
                        transforms.RandomRotation((90, 90)),
                        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize(IMG+32), transforms.CenterCrop(IMG),
                        transforms.RandomRotation((-90, -90)),
                        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
]

In [ ]:
# ── 6. Dataset + DataLoaders ──────────────────────────────────────────────────
class FundusDataset(Dataset):
    def __init__(self, dataframe, transform=None, do_crop=True):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform
        self.do_crop   = do_crop

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        if self.do_crop:
            img = crop_black_border_pil(img)
        if self.transform:
            img = self.transform(img)
        return img, int(row["y"])


ds_train = FundusDataset(train_df, transform=train_tfms)
ds_val   = FundusDataset(val_df,   transform=eval_tfms)
ds_test  = FundusDataset(test_df,  transform=eval_tfms)

train_loader = DataLoader(ds_train, batch_size=BATCH, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(ds_test,  batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train : {len(ds_train)} | Val : {len(ds_val)} | Test : {len(ds_test)}")

In [ ]:
# ── 7. Mixup ──────────────────────────────────────────────────────────────────
def mixup_data(x, y, alpha=0.3):
    """Mélange deux images et leurs labels — régularisation forte."""
    lam   = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx   = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
# ── 8. Class weights + Modèle EfficientNetV2-M ───────────────────────────────
cnt          = Counter(train_df["y"].tolist())
n_train      = len(train_df)
weights_list = [
    n_train / (num_classes * cnt[c]) if cnt.get(c, 0) > 0 else 1.0
    for c in range(num_classes)
]
weights_ce = torch.tensor(weights_list, dtype=torch.float32, device=device)
print("Class weights :", {le.classes_[i]: round(w, 4) for i, w in enumerate(weights_list)})

# EfficientNetV2-M — 54M params, +11 pts ImageNet vs ResNet34
model = models.efficientnet_v2_m(weights=models.EfficientNet_V2_M_Weights.IMAGENET1K_V1)
in_features = model.classifier[1].in_features   # 1280
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, num_classes),
)
model = model.to(device)

n_total = sum(p.numel() for p in model.parameters())
print(f"Paramètres totaux : {n_total:,}")

# Loss avec label smoothing (évite la sur-confiance)
criterion = nn.CrossEntropyLoss(weight=weights_ce, label_smoothing=0.1)
scaler    = GradScaler("cuda")

In [ ]:
# ── 9. Boucle d'entraînement ─────────────────────────────────────────────────
@torch.no_grad()
def evaluate(loader):
    model.eval()
    loss_sum, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with autocast("cuda"):
            logits = model(x)
            loss   = criterion(logits, y)
        loss_sum += loss.item() * x.size(0)
        correct  += (logits.argmax(1) == y).sum().item()
        n        += x.size(0)
    return loss_sum / max(n, 1), correct / max(n, 1)


def run_phase(epochs, lr, freeze_backbone, patience, label, use_cosine=False, use_mixup=False):
    # Gel / dégel
    for name, param in model.named_parameters():
        param.requires_grad = (not freeze_backbone) or ("classifier" in name)

    n_train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'='*65}")
    print(f"{label}")
    print(f"lr={lr}  |  params entraînables : {n_train_params:,}  |  mixup={use_mixup}")
    print(f"{'='*65}")

    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )

    if use_cosine:
        # Redémarre le cycle tous les 20 epochs — robuste avec early stopping
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=1, eta_min=lr * 0.01)
    else:
        scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=5)

    best_acc, best_state, wait = 0.0, None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        run_loss, n = 0.0, 0

        for x, y in train_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

            if use_mixup and MIXUP_ALPHA > 0:
                x, y_a, y_b, lam = mixup_data(x, y, alpha=MIXUP_ALPHA)

            optimizer.zero_grad(set_to_none=True)
            with autocast("cuda"):
                logits = model(x)
                if use_mixup and MIXUP_ALPHA > 0:
                    loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
                else:
                    loss = criterion(logits, y)

            scaler.scale(loss).backward()
            # Gradient clipping : évite les explosions de gradient
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            run_loss += loss.item() * x.size(0)
            n        += x.size(0)

        train_loss = run_loss / max(n, 1)
        val_loss, val_acc = evaluate(val_loader)

        if use_cosine:
            scheduler.step(epoch)          # CosineAnnealingWarmRestarts
        else:
            scheduler.step(val_acc)        # ReduceLROnPlateau

        lr_cur = optimizer.param_groups[0]["lr"]
        print(f"  Epoch {epoch:03d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | val_acc {val_acc:.4f} | lr {lr_cur:.2e}")

        if val_acc > best_acc:
            best_acc   = val_acc
            wait       = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stopping à l'epoch {epoch} (best val_acc={best_acc:.4f})")
                break

    return best_acc, best_state


# ── Phase 1 : backbone gelé, tête FC uniquement ──────────────────────────────
best_acc_p1, best_state_p1 = run_phase(
    epochs=EPOCHS_P1, lr=LR_P1,
    freeze_backbone=True, patience=EPOCHS_P1,  # pas d'early stop en phase 1
    label="PHASE 1 — backbone gelé, classifier uniquement",
    use_cosine=False, use_mixup=False,
)

# Recharger le meilleur état de phase 1 avant phase 2
model.load_state_dict(best_state_p1)

# ── Phase 2 : fine-tuning complet ────────────────────────────────────────────
best_acc_p2, best_state_p2 = run_phase(
    epochs=EPOCHS_P2, lr=LR_P2,
    freeze_backbone=False, patience=PATIENCE,
    label="PHASE 2 — backbone dégelé, fine-tuning complet + Mixup",
    use_cosine=True, use_mixup=True,
)

# Charger le meilleur état final
model.load_state_dict(best_state_p2)
print(f"\nMeilleur modèle rechargé — val_acc = {best_acc_p2:.4f}")

In [ ]:
# ── 10. Évaluation avec TTA ───────────────────────────────────────────────────
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    classification_report
)

model.eval()
tta_proba_sum = None
all_y = None

for t_idx, tfm in enumerate(tta_tfms_list):
    ds_tta = FundusDataset(test_df, transform=tfm, do_crop=True)
    ld_tta = DataLoader(ds_tta, batch_size=BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
    batch_proba, batch_y = [], []
    with torch.no_grad():
        for x, y in ld_tta:
            x = x.to(device, non_blocking=True)
            with autocast("cuda"):
                logits = model(x)
            proba = torch.softmax(logits, dim=1).cpu().numpy()
            batch_proba.append(proba)
            batch_y.append(y.numpy())
    proba_arr = np.concatenate(batch_proba, axis=0)
    if tta_proba_sum is None:
        tta_proba_sum = proba_arr
        all_y         = np.concatenate(batch_y, axis=0)
    else:
        tta_proba_sum += proba_arr
    print(f"TTA pass {t_idx+1}/{len(tta_tfms_list)} — done")

y_proba = tta_proba_sum / len(tta_tfms_list)
y_true  = all_y
y_pred  = y_proba.argmax(axis=1)

classes      = list(le.classes_)
target_names = classes

print("\n=== Scores globaux (avec TTA) ===")
print(f"Accuracy            : {accuracy_score(y_true, y_pred):.4f}")
print(f"Balanced accuracy   : {balanced_accuracy_score(y_true, y_pred):.4f}")
print(f"Precision (macro)   : {precision_score(y_true, y_pred, average='macro',    zero_division=0):.4f}")
print(f"Recall    (macro)   : {recall_score(   y_true, y_pred, average='macro',    zero_division=0):.4f}")
print(f"F1        (macro)   : {f1_score(       y_true, y_pred, average='macro',    zero_division=0):.4f}")
print(f"F1        (weighted): {f1_score(       y_true, y_pred, average='weighted', zero_division=0):.4f}")

print("\n=== Rapport par classe ===")
print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

In [ ]:
# ── 11. Matrice de confusion ──────────────────────────────────────────────────
label_indices = list(range(num_classes))
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, normalize, title in zip(
    axes,
    [None, "true"],
    ["Matrice de confusion (comptes)", "Matrice de confusion (normalisée)"],
):
    cm = confusion_matrix(y_true, y_pred, labels=label_indices, normalize=normalize)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
    disp.plot(ax=ax, colorbar=False, values_format=".2f" if normalize else "d")
    ax.set_title(title)
    ax.set_xlabel("Label prédit")
    ax.set_ylabel("Vrai label")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
# ── 12. Images mal classées ───────────────────────────────────────────────────
misclassified = np.where(y_true != y_pred)[0]
print(f"Images mal classées : {len(misclassified)} / {len(y_true)}")

df_test_reset = test_df.reset_index(drop=True)
for idx in misclassified[:6]:
    img_path   = df_test_reset.iloc[idx]["path"]
    true_label = classes[y_true[idx]]
    pred_label = classes[y_pred[idx]]
    confidence = y_proba[idx][y_pred[idx]]

    img = Image.open(img_path).convert("RGB")
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f"Vrai: {true_label}  |  Prédit: {pred_label}  ({confidence:.0%})")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# Sauvegarde des résultats EfficientNetV2-M pour comparaison
# ============================================================
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

results_eff = {
    "model"   : "EfficientNetV2-M",
    "acc"     : accuracy_score(y_true, y_pred),
    "bacc"    : balanced_accuracy_score(y_true, y_pred),
    "f1_macro": f1_score(y_true, y_pred, average='macro',    zero_division=0),
    "f1_w"    : f1_score(y_true, y_pred, average='weighted', zero_division=0),
    "y_true"  : y_true.copy(),
    "y_pred"  : y_pred.copy(),
    "y_proba" : y_proba.copy(),
}
print(f"EfficientNetV2-M  acc={results_eff['acc']:.4f}  f1_macro={results_eff['f1_macro']:.4f}")

# ── Téléchargement des poids RETFound (Color Fundus Photography) ─────────────
from huggingface_hub import hf_hub_download

print("\nTéléchargement des poids RETFound_cfp...")
retfound_ckpt = hf_hub_download(
    repo_id  = "rmaphoh/RETFound_cfp",
    filename = "RETFound_cfp.pth",
)
print(f"Poids sauvegardés : {retfound_ckpt}")

In [ ]:
# ============================================================
# RETFound — ViT-Large/16 pré-entraîné sur 1.6M rétinographies
# Beaucoup mieux qu'ImageNet pour des images de fond d'oeil
# ============================================================
import timm

IMG_RF   = 224    # résolution native de RETFound
BATCH_RF = 8      # ViT-L est plus grand (303M params) → batch réduit

# DataLoaders à 224 (résolution native RETFound)
eval_tfms_rf = transforms.Compose([
    transforms.Resize(IMG_RF + 32),
    transforms.CenterCrop(IMG_RF),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
train_tfms_rf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_RF, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    RandAugment(num_ops=2, magnitude=9),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
tta_tfms_rf = [
    eval_tfms_rf,
    transforms.Compose([transforms.Resize(IMG_RF+32), transforms.CenterCrop(IMG_RF),
                        transforms.RandomHorizontalFlip(p=1.0),
                        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize(IMG_RF+32), transforms.CenterCrop(IMG_RF),
                        transforms.RandomVerticalFlip(p=1.0),
                        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize(IMG_RF+32), transforms.CenterCrop(IMG_RF),
                        transforms.RandomRotation((90, 90)),
                        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
    transforms.Compose([transforms.Resize(IMG_RF+32), transforms.CenterCrop(IMG_RF),
                        transforms.RandomRotation((-90, -90)),
                        transforms.ToTensor(), transforms.Normalize(MEAN, STD)]),
]

ds_train_rf = FundusDataset(train_df, transform=train_tfms_rf)
ds_val_rf   = FundusDataset(val_df,   transform=eval_tfms_rf)
ds_test_rf  = FundusDataset(test_df,  transform=eval_tfms_rf)
train_loader_rf = DataLoader(ds_train_rf, batch_size=BATCH_RF, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader_rf   = DataLoader(ds_val_rf,   batch_size=BATCH_RF, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader_rf  = DataLoader(ds_test_rf,  batch_size=BATCH_RF, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# ── Modèle ViT-Large/16 ───────────────────────────────────────────────────────
model_rf = timm.create_model(
    'vit_large_patch16_224',
    pretrained=False,
    num_classes=num_classes,
    img_size=IMG_RF,
)

# ── Chargement des poids RETFound ─────────────────────────────────────────────
checkpoint    = torch.load(retfound_ckpt, map_location='cpu')
state_dict_rf = checkpoint['model']

# Retire les poids du décodeur MAE et du masque (pas utiles pour fine-tuning)
state_dict_rf = {k: v for k, v in state_dict_rf.items()
                 if not k.startswith('decoder') and k != 'mask_token'}
# Retire la tête de classification (tailles différentes)
state_dict_rf.pop('head.weight', None)
state_dict_rf.pop('head.bias',   None)

msg = model_rf.load_state_dict(state_dict_rf, strict=False)
print(f"Clés manquantes  : {msg.missing_keys}")    # attendu : head.weight, head.bias
print(f"Clés inattendues : {len(msg.unexpected_keys)} (décodeur MAE — normal)")

model_rf = model_rf.to(device)
print(f"\nParamètres totaux : {sum(p.numel() for p in model_rf.parameters()):,}")
print("RETFound chargé avec succès !")

In [ ]:
# ── Boucle d'entraînement générique (prend le modèle en paramètre) ────────────
def evaluate_rf(mdl, loader):
    mdl.eval()
    loss_sum, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with autocast('cuda'):
                logits = mdl(x)
                loss   = criterion(logits, y)
            loss_sum += loss.item() * x.size(0)
            correct  += (logits.argmax(1) == y).sum().item()
            n        += x.size(0)
    return loss_sum / max(n, 1), correct / max(n, 1)


def run_phase_rf(mdl, scaler_rf, train_ldr, val_ldr,
                  epochs, lr, head_only, patience,
                  label, use_cosine=False, use_mixup=False):
    # Gel / dégel — pour ViT la tête s'appelle 'head'
    for name, param in mdl.named_parameters():
        param.requires_grad = (not head_only) or ('head' in name)

    n_tp = sum(p.numel() for p in mdl.parameters() if p.requires_grad)
    print(f"\n{'='*65}")
    print(f"{label}")
    print(f"lr={lr}  |  params entraînables : {n_tp:,}  |  mixup={use_mixup}")
    print(f"{'='*65}")

    optimizer = AdamW(
        filter(lambda p: p.requires_grad, mdl.parameters()),
        lr=lr, weight_decay=1e-4
    )
    if use_cosine:
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, eta_min=lr * 0.01)
    else:
        scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    best_acc, best_state, wait = 0.0, None, 0

    for epoch in range(1, epochs + 1):
        mdl.train()
        run_loss, n = 0.0, 0
        for x, y in train_ldr:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            if use_mixup and MIXUP_ALPHA > 0:
                x, y_a, y_b, lam = mixup_data(x, y, MIXUP_ALPHA)
            optimizer.zero_grad(set_to_none=True)
            with autocast('cuda'):
                logits = mdl(x)
                loss   = (mixup_criterion(criterion, logits, y_a, y_b, lam)
                          if (use_mixup and MIXUP_ALPHA > 0) else criterion(logits, y))
            scaler_rf.scale(loss).backward()
            scaler_rf.unscale_(optimizer)
            nn.utils.clip_grad_norm_(mdl.parameters(), max_norm=1.0)
            scaler_rf.step(optimizer)
            scaler_rf.update()
            run_loss += loss.item() * x.size(0)
            n        += x.size(0)

        train_loss = run_loss / max(n, 1)
        val_loss, val_acc = evaluate_rf(mdl, val_ldr)

        if use_cosine: scheduler.step(epoch)
        else:          scheduler.step(val_acc)

        lr_cur = optimizer.param_groups[0]['lr']
        print(f"  Epoch {epoch:03d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} | val_acc {val_acc:.4f} | lr {lr_cur:.2e}")

        if val_acc > best_acc:
            best_acc   = val_acc
            wait       = 0
            best_state = {k: v.detach().cpu().clone() for k, v in mdl.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stopping à l'epoch {epoch} (best val_acc={best_acc:.4f})")
                break
    return best_acc, best_state


scaler_rf = GradScaler('cuda')

# ── Phase 1 : backbone gelé, tête seule ──────────────────────────────────────
best_acc_rf_p1, best_state_rf_p1 = run_phase_rf(
    model_rf, scaler_rf, train_loader_rf, val_loader_rf,
    epochs=10, lr=1e-3, head_only=True, patience=10,
    label='RETFOUND — PHASE 1 : backbone gelé, head uniquement',
    use_cosine=False, use_mixup=False,
)
model_rf.load_state_dict(best_state_rf_p1)

# ── Phase 2 : fine-tuning complet ────────────────────────────────────────────
best_acc_rf_p2, best_state_rf_p2 = run_phase_rf(
    model_rf, scaler_rf, train_loader_rf, val_loader_rf,
    epochs=300, lr=5e-5, head_only=False, patience=25,
    label='RETFOUND — PHASE 2 : fine-tuning complet + Mixup',
    use_cosine=True, use_mixup=True,
)
model_rf.load_state_dict(best_state_rf_p2)
print(f"\nRETFound meilleur modèle rechargé — val_acc = {best_acc_rf_p2:.4f}")

In [ ]:
# ── TTA RETFound ──────────────────────────────────────────────────────────────
model_rf.eval()
tta_sum_rf, all_y_rf = None, None

for t_idx, tfm in enumerate(tta_tfms_rf):
    ds_tta_rf = FundusDataset(test_df, transform=tfm, do_crop=True)
    ld_tta_rf = DataLoader(ds_tta_rf, batch_size=BATCH_RF, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
    bp, by = [], []
    with torch.no_grad():
        for x, y in ld_tta_rf:
            x = x.to(device, non_blocking=True)
            with autocast('cuda'):
                logits = model_rf(x)
            bp.append(torch.softmax(logits, dim=1).cpu().numpy())
            by.append(y.numpy())
    arr = np.concatenate(bp, axis=0)
    if tta_sum_rf is None:
        tta_sum_rf = arr
        all_y_rf   = np.concatenate(by)
    else:
        tta_sum_rf += arr
    print(f"TTA pass {t_idx+1}/{len(tta_tfms_rf)} — done")

y_proba_rf = tta_sum_rf / len(tta_tfms_rf)
y_true_rf  = all_y_rf
y_pred_rf  = y_proba_rf.argmax(axis=1)

results_rf = {
    'model'   : 'RETFound (ViT-L/16)',
    'acc'     : accuracy_score(y_true_rf, y_pred_rf),
    'bacc'    : balanced_accuracy_score(y_true_rf, y_pred_rf),
    'f1_macro': f1_score(y_true_rf, y_pred_rf, average='macro',    zero_division=0),
    'f1_w'    : f1_score(y_true_rf, y_pred_rf, average='weighted', zero_division=0),
    'y_true'  : y_true_rf.copy(),
    'y_pred'  : y_pred_rf.copy(),
    'y_proba' : y_proba_rf.copy(),
}

print('\n=== Rapport RETFound ===')
print(classification_report(y_true_rf, y_pred_rf, target_names=classes, zero_division=0))

# ── Tableau comparatif ────────────────────────────────────────────────────────
print('\n' + '='*60)
print(f"{'Modèle':<22} {'Accuracy':>10} {'Bal.Acc':>10} {'F1 macro':>10} {'F1 weighted':>12}")
print('-'*60)
for r in [results_eff, results_rf]:
    print(f"{r['model']:<22} {r['acc']:>10.4f} {r['bacc']:>10.4f} {r['f1_macro']:>10.4f} {r['f1_w']:>12.4f}")
print('='*60)

# ── Matrices de confusion côte à côte ─────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
configs = [
    (results_eff['y_true'], results_eff['y_pred'], 'EfficientNetV2-M (comptes)',     axes[0,0], None),
    (results_eff['y_true'], results_eff['y_pred'], 'EfficientNetV2-M (normalisée)',  axes[0,1], 'true'),
    (results_rf['y_true'],  results_rf['y_pred'],  'RETFound ViT-L/16 (comptes)',    axes[1,0], None),
    (results_rf['y_true'],  results_rf['y_pred'],  'RETFound ViT-L/16 (normalisée)', axes[1,1], 'true'),
]
for yt, yp, title, ax, norm in configs:
    cm = confusion_matrix(yt, yp, labels=list(range(num_classes)), normalize=norm)
    ConfusionMatrixDisplay(cm, display_labels=classes).plot(
        ax=ax, colorbar=False, values_format='.2f' if norm else 'd')
    ax.set_title(title)
    ax.set_xlabel('Label prédit')
    ax.set_ylabel('Vrai label')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.suptitle('Comparaison EfficientNetV2-M vs RETFound', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()